# Hybrid vision + spatial reasoning on Amazon Bedrock

**Use case.** Match the names printed on a scanned page to the photographs on that page. The example here is a yearbook layout: each page contains photos of people and printed names that belong to those people, but the name-to-photo link only exists in the page layout itself — not in the file.

**Pipeline.** Two Bedrock calls per page:

1. **Amazon Nova 2 Lite** — natively reads interleaved text and images. One InvokeModel call returns photo bounding boxes, the visible names with their positions, and page-level metadata (title, category, summary). All bounding boxes use a 0–1000 normalized grid.
2. **Claude Sonnet 4.6 with adaptive thinking** — receives the original page image plus Nova's structured output and decides which name maps to which face based on spatial layout (caption above/below, columns, group-photo captions).

Why split the work. Nova handles the high-volume native multimodal extraction in one call. Claude is invoked once per page for the reasoning step that benefits from extra thinking. Each stage can be tuned or swapped independently.

**What you will run.** Three synthetic yearbook pages — a portrait grid, a mixed-layout floor show page, and a group-photo academic page — end-to-end through both stages, with visualizations of the matched names and faces.

## 1. Prerequisites

- An AWS account with access to Amazon Bedrock.
- Model access enabled in your Amazon Bedrock region for `us.amazon.nova-2-lite-v1:0` and `us.anthropic.claude-sonnet-4-6`.
- An AWS Identity and Access Management (AWS IAM) role or user with `bedrock:InvokeModel` and `bedrock:Converse` permission on those two models.
- Python 3.10 or later, plus `boto3` and `Pillow` (installed in the next cell).

## 2. Setup

Install dependencies and create the Amazon Bedrock Runtime client. The notebook uses the `us-west-2` region by default; change `AWS_REGION` in the configuration cell below if your access lives elsewhere.

If you are running on Amazon SageMaker Studio or a Notebook Instance, the dependencies installed in the next cell are already available.

In [ ]:
%pip install --quiet boto3 Pillow

In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, JSON, display

from utils import (
    CLAUDE_MODEL_ID,
    NOVA_MODEL_ID,
    extract_photos_and_names,
    load_image_bytes,
    make_bedrock_client,
    match_names_to_faces,
    run_pipeline,
    visualize_result,
)

AWS_REGION = "us-west-2"
SAMPLES_DIR = Path("samples")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

bedrock = make_bedrock_client(AWS_REGION)
print("Nova model :", NOVA_MODEL_ID)
print("Claude model:", CLAUDE_MODEL_ID)
print("Region     :", AWS_REGION)

## 2. Sample pages

Three synthetic yearbook pages live in `samples/`. The names printed on these pages are fictional and were generated specifically for this sample — no real student data is used.

| File | Layout |
|---|---|
| `page_001_portrait_grid.png` | 4×5 portrait grid with one name printed under each headshot |
| `page_002_floor_show.png` | Mixed layout: one group photo (no caption) plus several candid/portrait photos with italic captions |
| `page_003_decathlon.png` | Single group photo with a caption listing every person back-row-then-front-row |

In [ ]:
samples = sorted(SAMPLES_DIR.glob("page_*.png"))
for p in samples:
    print(f"{p.name:40s}  {p.stat().st_size/1024:.0f} KB")

In [ ]:
from IPython.display import Markdown
# Render with alt text so screen readers can describe the image.
display(Markdown(
    f'![Synthetic yearbook portrait grid sample page used as input to Stage 1.]({samples[0]})'
))

## 3. Stage 1 — Nova 2 Lite extracts photos, names, and metadata

One InvokeModel call returns:

- `photos`: list of bounding boxes plus `type` (`portrait` / `group_photo` / `candid`), category, and a one-line summary.
- `names`: list of printed personal names with their bounding boxes.
- `page_title`, `page_category`, `page_summary`: page-level metadata that doubles as the second use case (search indexing, content tagging) without an extra API call.

All bounding boxes are on a 0–1000 normalized grid for both axes. The same coordinate space carries through to Stage 2, so no conversion is needed between calls.

In [ ]:
image_bytes = load_image_bytes(samples[0])
extraction = extract_photos_and_names(bedrock, image_bytes)

print("Page title    :", extraction["page_title"])
print("Page category :", extraction["page_category"])
print("Page summary  :", extraction["page_summary"])
print("Photos detected:", len(extraction["photos"]))
print("Names detected :", len(extraction["names"]))
print("Token usage   :", extraction["_usage"])

In [ ]:
display(JSON({"first_three_photos": extraction["photos"][:3], "first_three_names": extraction["names"][:3]}))

## 4. Stage 2 — Claude Sonnet 4.6 matches names to faces

Claude receives the original page image plus Nova's photos and names. Adaptive thinking is enabled so Claude allocates more reasoning to mixed-layout pages than to clean grids. The effort is set to `high` to make sure Claude always reasons through the spatial layout (the API call is identical to a regular Converse call apart from `additionalModelRequestFields`).

In [ ]:
matching = match_names_to_faces(
    bedrock,
    image_bytes,
    extraction["photos"],
    extraction["names"],
)

print("Associations    :", len(matching["associations"]))
print("Unmatched names :", matching["unmatched_names"])
print("Unmatched faces :", matching["unmatched_face_indices"])
print("Thinking chars  :", len(matching["thinking_text"] or ""))
print("Token usage     :", matching["_usage"])

In [ ]:
for assoc in matching["associations"][:8]:
    print(
        f"{assoc['name']:25s} -> face {assoc['face_idx']:>2}  "
        f"({assoc['match_type']}, conf {assoc['confidence']:.2f})\n"
        f"    reason: {assoc.get('reasoning', '')}"
    )

## 5. End-to-end on all three sample pages

`run_pipeline` runs Stage 1 and Stage 2 back to back and returns one combined result. `visualize_result` draws the photo bounding boxes plus a colored line from each printed name to the matched face.

In [ ]:
all_results = {}

for sample in samples:
    print(f"=== {sample.name} ===")
    result = run_pipeline(bedrock, sample)

    summary = {
        "page_title": result["page_title"],
        "page_category": result["page_category"],
        "photos": len(result["photos"]),
        "names": len(result["names"]),
        "associations": len(result["associations"]),
        "unmatched_names": result["unmatched_names"],
        "unmatched_face_indices": result["unmatched_face_indices"],
        "nova_tokens": result["usage"]["nova"],
        "claude_tokens": result["usage"]["claude"],
    }
    print(json.dumps(summary, indent=2))

    base = sample.stem
    viz_path = visualize_result(sample, result, RESULTS_DIR / f"{base}_links.jpg")
    json_path = RESULTS_DIR / f"{base}_result.json"
    serializable = {k: v for k, v in result.items() if k != "thinking_text"}
    serializable["thinking_chars"] = len(result["thinking_text"] or "")
    json_path.write_text(json.dumps(serializable, indent=2))
    all_results[sample.name] = result
    print(f"  visualization -> {viz_path}")
    print(f"  raw output    -> {json_path}\n")

In [ ]:
from IPython.display import Markdown
for sample in samples:
    viz = RESULTS_DIR / f"{sample.stem}_links.jpg"
    print(viz.name)
    # Render with alt text describing what the visualization shows.
    display(Markdown(
        f'![Visualization of {sample.stem}: photo bounding boxes with colored lines linking each printed name to the matched face.]({viz})'
    ))

## 6. Use case 2 — page-level metadata in the same call

The Nova call already returned `page_title`, `page_category`, and `page_summary`. That is enough to build a search index, a per-event filter, or a table of contents across hundreds of pages without any extra API call.

In [ ]:
for name, result in all_results.items():
    print(f"{name}")
    print(f"  title    : {result['page_title']}")
    print(f"  category : {result['page_category']}")
    print(f"  summary  : {result['page_summary']}")
    print()

## 7. Notes on cost, latency, and tuning

- **Cost.** Nova 2 Lite supports a fixed per-image token tier for image and document-page inputs, which makes Stage 1 cost predictable regardless of page resolution. Confirm the current rate on the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/) before you size a workload. Claude's adaptive-thinking output dominates per-page cost on complex pages.
- **Latency.** Stage 1 typically returns in a few seconds. Stage 2 is the longer call because of the image input plus adaptive reasoning; expect roughly 20–30 seconds on complex pages.
- **Effort knob.** `output_config.effort` accepts `low`, `medium`, `high`, or `max`. Lowering effort to `medium` skips reasoning on simple grids and saves output tokens; raising to `max` (Opus only) gives Claude the deepest reasoning budget.
- **Batch inference.** Both Nova 2 Lite and Claude support [Amazon Bedrock Batch Inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-inference.html) at a 50% discount for workloads that can run asynchronously.
- **Prompt caching.** The Nova extraction prompt is identical across pages and is a good candidate for [prompt caching](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html) when running at volume.
- **Adapt the prompts.** `utils.py` exposes `NOVA_SYSTEM_PROMPT`, `NOVA_INSTRUCTION`, and `CLAUDE_SPATIAL_PROMPT_TEMPLATE`. Edit them in place to handle other document layouts (real estate listings, product catalogs, magazine spreads). The rest of the pipeline stays the same.

## 8. Clean up

This pattern is fully serverless. There are no Amazon Bedrock endpoints, Amazon SageMaker AI instances, or persistent storage to delete. The notebook writes outputs to the local `results/` directory; delete this directory if you no longer need the visualization JPEGs and JSON files. If you uploaded sample pages to Amazon Simple Storage Service (Amazon S3) to run this at scale, remove the bucket or objects when you finish. Warning: deleting Amazon S3 objects is permanent and cannot be undone — back up any data you need to keep before deletion.

## 9. Conclusion

Two Amazon Bedrock calls are enough to map printed names to faces on a scanned page. Amazon Nova 2 Lite carries the native multimodal extraction in a single call, and Claude Sonnet 4.6 with adaptive thinking handles the spatial reasoning step that benefits from extra reasoning. Keeping the two stages on the same 0–1000 coordinate space removes glue code between calls, and each stage stays independently tunable. As next steps, swap the synthetic samples in `samples/` for your own documents, adapt the prompts in `utils.py` to match your layout, and rerun this notebook end to end to verify per-page accuracy on your own data.